# 演習4 解答編 ―― 待ち合わせ

> まず `ex04_queue_wait.ipynb` を自分で解いてから読んでください。
> このノートには、答えを**自分で確かめるためのコード**も入っています。上から順に ▶ を押してください。

## 発展課題1 の解答 ―― 眠る時間を「作る時間」に合わせても駄目な理由

うまくいかない場面は、少なくとも4つあります。

**① 作る時間がばらつく**

「平均200ms」でも、実際には 150ms のときも 400ms のときもあります。
平均に合わせると、速いときは遅れ、遅いときは無駄に確認します。
本番で扱う推論処理も、画面に映っているものによって所要時間が変わります。

**② 作る側の速さが途中で変わる**

動画の内容、マシンの負荷、他のスレッドの混み具合で変わります。
プログラムを書いた時点で決めた定数が、実行中ずっと正しいという保証はありません。

**③ 受け取る側が複数いる**

2人で分担していれば、1人あたりの間隔は倍になります。
人数を変えるたびに定数を直すことになります。

**④ 段が増えると、遅れが足し算になる**

パイプラインが4段あって、各段が 10ms のポーリングをしていたら、
最悪で 40ms の遅れが積み上がります。1段だけ見ていては気づけません。

そして、いちばん大きな問題はこれです。

> **「相手が何msで作るか」を知っていないと決められない、ということ自体が設計上の欠陥。**

受け取る側が作る側の内部事情に依存してしまうと、片方を直すたびにもう片方も直すはめになります。
`condition_variable` を使えば、この定数そのものが要らなくなります。

## 発展課題2 の解答 ―― `notify_one` と `notify_all`

**受け取る係が1人なら、違いはありません。** 待っている人が1人しかいないので、
「1人起こす」も「全員起こす」も同じことです。

**複数いると差が出ます。** キューに1個入れたとき、実際に動けるのは1人だけです。
`notify_all()` で10人起こしても、1人が取ったあとの9人は
「まだ空だった」と分かって眠り直すだけです。

この「全員起きて、1人以外は無駄足」を **サンダリングハード（thundering herd、群れの暴走）**
と呼びます。次のセルで実際に数えます。

In [ ]:
%%writefile ans04a.cpp
#include <iostream>
#include <thread>
#include <vector>
#include <queue>
#include <mutex>
#include <condition_variable>
#include <chrono>
using namespace std::chrono;

std::queue<int> q;
std::mutex mtx;
std::condition_variable can_pop;
bool done;
long checks;                      // 述語を評価した回数 ＝ 起こされた回数 + 最初の1回

void consumer() {
    while (true) {
        std::unique_lock<std::mutex> lk(mtx);
        can_pop.wait(lk, [] { checks++; return !q.empty() || done; });
        if (q.empty()) return;                    // done で起こされた＝もう来ない
        q.pop();
    }
}

void run(bool use_all, int nconsumer) {
    while (!q.empty()) q.pop();
    done = false; checks = 0;

    std::vector<std::thread> cs;
    for (int i = 0; i < nconsumer; i++) cs.emplace_back(consumer);
    std::this_thread::sleep_for(milliseconds(100));      // 全員が待ちに入るのを待つ

    for (int i = 0; i < 20; i++) {                       // 20個入れる
        { std::lock_guard<std::mutex> g(mtx); q.push(i); }
        if (use_all) can_pop.notify_all(); else can_pop.notify_one();
        std::this_thread::sleep_for(milliseconds(5));
    }
    { std::lock_guard<std::mutex> g(mtx); done = true; }
    can_pop.notify_all();
    for (auto& t : cs) t.join();

    std::cout << "受け取る係 " << nconsumer << "人 / "
              << (use_all ? "notify_all" : "notify_one")
              << " : 述語を評価した回数 = " << checks << "\n";
}

int main() {
    std::cout << "20個のデータを入れる。数字が小さいほど無駄が少ない。\n\n";
    run(false, 1);
    run(true,  1);
    std::cout << "\n";
    run(false, 10);
    run(true,  10);
    return 0;
}

In [ ]:
!g++ -std=c++17 -pthread ans04a.cpp -o ans04a && ./ans04a

数字は「述語を評価した回数」です。

- 1人のとき ⇒ `notify_one` も `notify_all` も同じ（42回）
- 10人のとき ⇒ `notify_one` は 60回、`notify_all` は **240回**。約4倍の無駄

### 述語が評価される場面は3通りある

まず、この数がどこから出てくるのかを押さえます。`consumer()` を見てください。

```cpp
void consumer() {
    while (true) {
        std::unique_lock<std::mutex> lk(mtx);
        can_pop.wait(lk, [] { checks++; return !q.empty() || done; });
        if (q.empty()) return;
        q.pop();
    }                    // ← ここから while の先頭に戻り、もう一度 wait を呼ぶ
}
```

`wait` が呼ばれるたびに、述語は必ず1回は評価されます。場面は3つです。

- **① `wait` を呼んだ直後、眠る前**
  `wait(lk, 述語)` は、**まず述語を確かめます。すでに真なら、眠りません。**
  4-3 で見た「3つの仕事」の1番目です
- **② 起こされて、鍵を取り直した直後**
  `notify_one` なら1人、`notify_all` なら待っている全員が、ここで1回ずつ評価します
- **③ 取り出したあと、ループを1周してまた `wait` を呼んだとき**
  `q.pop()` した本人が while の先頭へ戻ってきます。このときキューはもう空なので偽 ⇒ また眠る

**①と③は、同じ行を通っているだけで中身は同じ**です。
「`wait` を呼べば、必ず1回は述語を見る」というだけの話です。

### 数字の内訳

この3つで、出てきた数がぴったり説明できます。

**受け取る係が10人 / `notify_one`（= 60回）**

```
最初に10人が wait に入る（①）        : 10 人 × 1回 = 10
データ1個ごと
   起こされた1人が評価（②、真）      :  1
   その人が取り出して wait に戻る（③）:  1   ← このとき q は空なので偽
                                       計 2回 × 20個 = 40
最後に done を notify_all（②）       : 10 人 × 1回 = 10
                                                    合計 60
```

**受け取る係が10人 / `notify_all`（= 240回）**

```
最初に10人が wait に入る（①）        : 10
データ1個ごと
   起こされた10人全員が評価（②）      : 10   ← 真になるのは1人だけ。9人は眠り直す
   取れた1人が wait に戻る（③）       :  1
                                       計 11回 × 20個 = 220
最後に done を notify_all（②）       : 10
                                                    合計 240
```

**受け取る係が1人（= 42回、どちらも同じ）**

```
最初に wait に入る（①）              :  1
1個ごとに 2回 × 20個                 : 40
最後の done                          :  1
                                                    合計 42
```

待っている人が1人しかいないので、「1人起こす」と「全員起こす」が同じ動作になります。
だから 42 / 42 で一致します。

### 式にすると

```
notify_one : 2 × 件数 + 2 × 人数
notify_all : (人数 + 1) × 件数 + 2 × 人数
```

**無駄の部分が `人数 × 件数` に比例している**のがポイントです。
人数が10倍になれば無駄も10倍、データが1000個流れれば1000倍。
パイプラインのように何万フレームも流し続けるものでは、ここが効いてきます。

しかも、増えるのは評価回数だけではありません。
9人が「起きて、**鍵を取り直して**、条件を見て、また眠る」わけですから、
4-4 で見た**鍵の取り合い**も9回ぶん余計に発生しています。

> **1個入れたのなら、起こすのは1人でよい。**

### 誰が起きるかは、指定できない

ここで当然わいてくる疑問があります。**10人のうち、誰が起きるのでしょうか。**

**指定できません。そして、決まってもいません。**

`notify_one()` の仕様は「待っている中の**1つを起こす**」だけで、
**どれを起こすかは規定されていません。** 優先順位もなければ、
「先に待ち始めた人が先に起きる」という保証（FIFO）もありません。
実装によっては待ち行列に近い順で起きることが多いのですが、**当てにしてはいけません。**

さらに厳しいことを言うと、**「起こされた人 ＝ 受け取る人」ですらありません。**
4-4 で見たとおり、起こされた人は鍵を取り直すまで何もできません。その隙に、
`wait` で眠っていなかった別の消費者がたまたま `pop()` を呼びに来て、
先に鍵を取ってしまうことがあります。

つまり `notify_one` が保証しているのは、これだけです。

> **「待っている人が1人以上いるなら、そのうち1人は起きる」。**
> **誰が起きるかも、その人が仕事を取れるかも、保証しない。**

### 特定の相手に渡したいときは、構造を変える

「この仕事はスレッド3に渡したい」という場合、条件変数を1本で共有している設計では無理です。
**渡し方のほうを変えます。**

- **スレッドごとに条件変数を持たせる** … 起こしたい相手の `cv` だけを `notify` する
- **スレッドごとにキューを分ける** … 渡したい相手のキューに `push` する

後者が、**演習8で扱う「担当ごとにレーンを分ける」方式**です。
Read が「Aに0番、Bに1番…」と**行き先を指定して**配る形になります。

ただし代償があります。行き先を固定した瞬間、
**重い仕事が片方に偏っても、暇なほうは手伝えなくなります。**
演習8の発展課題5で実測しますが、偏りがあると FPS が3割以上落ちることもあります。

**逆に言えば、「誰が受け取るか指定できない」ことは、たいていの場合むしろ利点です。**
1本のキューを取り合う形では、手が空いた人が次を取るので、**勝手に仕事が均されます。**

行き先を指定したくなったら、まず「本当に指定する必要があるのか」を疑ってください。
指定が要るのは、**担当者が交換可能でない**とき ―― たとえば

- スレッドごとに使う**ハードウェアが違う**（0番のアクセラレータ担当・1番担当）
- スレッドごとに**持っている状態が違う**（このスレッドだけが特定のモデルを読み込んでいる）

といった場合だけです。交換可能なら、指定しないほうが速くなります。

### では `notify_all` はいつ使うのか

「**変化によって、複数の人がいっぺんに進めるようになる**」ときです。代表例が2つあります。

- **終了を知らせるとき**（発展課題3）。「もう来ない」は待っている全員に関係します
- **待っている条件が人によって違うとき**。誰が進めるようになったか分からないので、
  全員に確かめさせるしかありません

判断の基準は「起こした人数のうち、実際に進めるのは何人か」です。
1人しか進めないなら `notify_one`、全員が進めるなら `notify_all` です。

### 内訳に出てきた「最後の done」について

上の数え上げに出てくる「最後に done を notify_all」は、`run()` の末尾の2行です。

```cpp
    for (int i = 0; i < 20; i++) {
        { std::lock_guard<std::mutex> g(mtx); q.push(i); }
        if (use_all) can_pop.notify_all(); else can_pop.notify_one();   // ← ここは切り替わる
        std::this_thread::sleep_for(milliseconds(5));
    }
    { std::lock_guard<std::mutex> g(mtx); done = true; }    // ← 切り替えとは無関係
    can_pop.notify_all();                                   // ← ここは必ず notify_all
```

**`use_all` の切り替えとは無関係に、ここだけは必ず `notify_all` です。**
`notify_one` にすると1人しか終われず、残り9人は永久に眠ったままになります。

なぜそうなのか、そしてこの2行をどう書くべきかは、**発展課題3**で扱います。

## 発展課題3 の解答 ―― 「もう作らない」をどう伝えるか

個数を決め打ちできないなら、**終了フラグ**を1つ用意します。

```cpp
bool done = false;                                  // 鍵で守る共有データ
...
can_pop.wait(lk, [] { return !q.empty() || done; }); // ← 条件を2つに
```

述語が「**データが来た、または、もう来ない**」の2条件になるのがポイントです。
`wait` から戻ったあとに、どちらで起きたのかを見分けます。

```cpp
if (q.empty()) return;      // 空なのに起きた ＝ done で起こされた ＝ 終わり
```

### 作る側の最後の2行

そして、作る側の最後はこの2行になります。**この演習でいちばん重要な2行です。**

```cpp
{ std::lock_guard<std::mutex> g(mtx); done = true; }   // ① 鍵の中で書き換える
can_pop.notify_all();                                  // ② 鍵の外で、全員を起こす
```

短いのに、**3つの決まりが全部入っています。** どれを外しても壊れます。

**決まり1 : `done` は鍵の中で書き換える**

`done` は述語に出てくる**共有データ**です。
「`bool` 1個だから大丈夫だろう」は通用しません（演習2の発展課題2）。
鍵の外で書き換えると、読む側にいつ見えるかの保証がなくなります。

**決まり2 : `notify` は鍵の外で呼ぶ**

鍵を握ったまま起こすと、起こされた側が鍵を取り直せずにもう一度眠ります
（演習3の発展課題4の hurry up and wait）。だから `{ }` で区切って、
**解錠してから通知します。**

**決まり3 : `notify_one` ではなく `notify_all`**

待っている人が3人いれば、**3人とも終わらせなければなりません。**
`notify_one` では1人しか起きず、残り2人は永久に眠ったまま。
`join()` が返らず、プログラムが終わらなくなります。

発展課題2の基準に当てはめると、こうなります。

- データを1個入れた ⇒ 進めるのは1人 ⇒ **`notify_one`**
- 「もう来ない」と宣言した ⇒ **待っている全員**が進める ⇒ **`notify_all`**

### 一般化すると

この2行は、`done` に限った話ではありません。

> **述語に出てくる変数を変えたら、必ず対応する条件変数を起こす。**
> **変えるのは鍵の中で、起こすのは鍵の外で。**
> **`one` と `all` は「何人が進めるようになるか」で選ぶ。**

```cpp
{ std::lock_guard<std::mutex> g(mtx); 述語に出てくる何かを変える; }
cv.notify_one();    // 1人だけ進めるようになるなら
cv.notify_all();    // 全員が進めるようになるなら
```

**この形は、このあとずっと出てきます。**

- 演習5 … `push` / `pop` が、`q_` を変えたあとに相手側を起こす
- 演習5の発展課題6 … `capacity_` を変えたあとに `can_push_.notify_all()`
- 演習9 … キューを `close()` したあとに `notify_all()`

いずれも同じ2行です。**ここで形として覚えてしまってください。**

次のセルで動かします。

In [ ]:
%%writefile ans04b.cpp
#include <iostream>
#include <thread>
#include <vector>
#include <queue>
#include <mutex>
#include <condition_variable>
#include <chrono>
using namespace std::chrono;

std::queue<int> q;
std::mutex mtx;
std::condition_variable can_pop;
bool done = false;               // ← 「もう作らない」を伝えるフラグ（鍵で守る）

void consumer(int id) {
    while (true) {
        int v;
        {
            std::unique_lock<std::mutex> lk(mtx);
            can_pop.wait(lk, [] { return !q.empty() || done; });   // ← 条件を2つに
            if (q.empty()) {                     // 空なのに起きた＝done
                std::cout << "  受け取る係" << id << " : もう来ないので終了\n";
                return;
            }
            v = q.front(); q.pop();
        }
        std::cout << "  受け取る係" << id << " : " << v << " を受け取った\n";
    }
}

int main() {
    std::thread c0(consumer, 0), c1(consumer, 1);

    for (int i = 1; i <= 4; i++) {
        std::this_thread::sleep_for(milliseconds(100));
        { std::lock_guard<std::mutex> g(mtx); q.push(i); }
        can_pop.notify_one();
    }

    std::cout << "  作る係 : もう作らない\n";
    { std::lock_guard<std::mutex> g(mtx); done = true; }
    can_pop.notify_all();                        // ← 全員に知らせる。notify_one では1人しか起きない

    c0.join(); c1.join();
    std::cout << "  すべてのスレッドが join() できた\n";
    return 0;
}

In [ ]:
!g++ -std=c++17 -pthread ans04b.cpp -o ans04b && ./ans04b

2人とも「もう来ないので終了」で抜け、`join()` まで到達しています。

もう1つ、コードを見るときに気をつける点があります。

- **残っているデータを捨ててはいけません。** 述語は `!q.empty() || done` の順です。
  `done` が立っていても、**キューに残っていれば先に取り出します。**
  もし `if (done) return;` を先に書いてしまうと、
  閉じた瞬間に残りを全部捨てることになります

> **「もう来ない」と「もう空だ」は別のこと。**
> **終わってよいのは、その両方がそろったときだけ。**

キューを安全に終わらせる話は、**演習9**でもう一度きちんと扱います。

## 発展課題4 の解答 ―― 述語の中で共有データを読んでよいのか

**競合しません。安全です。**

理由は、**述語が評価されるときは必ず鍵が握られている**からです。
4-3 で見た `wait` の3つの仕事を、もう一度並べます。

1. 述語を確かめる ← このとき鍵は**呼び出した側がすでに握っている**（`unique_lock lk(mtx);`）
2. 偽なら、鍵を開けて眠る
3. 起こされたら、**鍵を取り直してから**述語を確かめる ← ここでも鍵は握られている

つまり述語の中は、いつでも「鍵の中」です。だから `q.empty()` を読んで構いません。

**逆に、やってはいけないこと**があります。

```cpp
can_pop.wait(lk, [] {
    std::lock_guard<std::mutex> g(mtx);    // ← これはいけない
    return !q.empty();
});
```

すでに握っている鍵を、もう一度かけようとしています。
演習3の発展課題1で見た**二重ロック＝デッドロック**です。
「共有データを読むから鍵をかけなきゃ」と反射的に書くと、こうなります。

> **述語の中はすでに鍵の中。追加で鍵をかけてはいけない。**

## 発展課題5 の解答 ―― `notify_one()` を書き忘れると

異常終了はしません。**プログラムが静かに固まります。**
待っている側は誰にも起こされないので、永久に眠り続けます。

厄介なのは、**書き忘れても動いてしまう場合がある**ことです。
次のセルで、動く場合と固まる場合を並べて確かめます。5秒で強制終了させます。

In [ ]:
%%writefile ans04c.cpp
#include <iostream>
#include <thread>
#include <queue>
#include <mutex>
#include <condition_variable>
#include <chrono>
using namespace std::chrono;

std::queue<int> q;
std::mutex mtx;
std::condition_variable can_pop;

void consumer() {
    std::unique_lock<std::mutex> lk(mtx);
    can_pop.wait(lk, [] { return !q.empty(); });
    std::cout << "  受け取った : " << q.front() << "\n" << std::flush;
    q.pop();
}

int main() {
    std::cout << "【ケースA】先にデータを入れてから、受け取る係を起動する\n" << std::flush;
    { std::lock_guard<std::mutex> g(mtx); q.push(1); }
    // notify_one() を呼んでいない
    std::thread a(consumer);
    a.join();
    std::cout << "  → notify を呼んでいないのに、動いてしまった\n\n" << std::flush;

    std::cout << "【ケースB】受け取る係を先に起動し、あとからデータを入れる\n" << std::flush;
    std::thread b(consumer);
    std::this_thread::sleep_for(milliseconds(200));
    { std::lock_guard<std::mutex> g(mtx); q.push(2); }
    // ここでも notify_one() を呼んでいない
    std::cout << "  データは入れた。しかし誰も起こしていない...\n" << std::flush;
    b.join();
    std::cout << "  ここには到達しない\n";
    return 0;
}

In [ ]:
!g++ -std=c++17 -pthread ans04c.cpp -o ans04c
!timeout 5 ./ans04c; echo "終了コード=$? （124 なら固まった）"

- **ケースA は動きます。** すでにキューにデータがあるので、`wait` は
  「述語が最初から真」と判断して**眠らずにそのまま通過**します。
  起こしてもらう必要がありません
- **ケースB は固まります。** 先に眠ってしまったので、誰かが起こさないかぎり出てこられません

つまりこのバグは、**タイミング次第で出たり出なかったりします。**

- データが十分に溜まっている状態でテストすると、通ってしまう
- 負荷が軽くて作る側が先行しているうちは、通ってしまう
- 本番で受け取る側が追いついた瞬間に、止まる

演習2で見た競合バグと、性質はまったく同じです。

> **「動いたから正しい」が通用しない。**

なお、`condition_variable` には **spurious wakeup**（誰も起こしていないのに起きる現象）が
あるので、運が良ければ勝手に動き出すこともあります。
しかしそれは**当てにできる動作ではありません**。

`push` と `notify` は必ずセットで書く、と決めてしまうのが確実です。
演習5で組み立てるキューでは、この2つがクラスの中に閉じ込められるので、
使う側が書き忘れることはできなくなります。

---

## 参考：本番のプログラムでは

ハッカソンで読むプログラムのキューは、まさにこの形をしています。

- 鍵が1つ、`condition_variable` が2つ（「取り出せる」用と「入れられる」用）
- `wait` にはどちらも述語が付いている
- `push` した直後に `notify` している

演習5で、このキューを最初から最後まで自分で組み立てます。